<a href="https://colab.research.google.com/github/barrevivo299-design/SMS-Spam-Classification-NaiveBayes/blob/main/SMS_Spam_Classification_NaiveBayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, f1_score

# 1. Load Dataset
url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'text'])

# Convert labels to binary: ham -> 0, spam -> 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

print("--- Dataset Sample ---")
print(df.head())

# 2. Train-Test Split (80% Train, 20% Test)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['text'], df['label_num'], test_size=0.2, random_state=42, stratify=df['label_num']
)

# 3. Feature Engineering (CountVectorizer)
vectorizer = CountVectorizer(stop_words='english', lowercase=True)
X_train = vectorizer.fit_transform(X_train_raw).toarray()
X_test = vectorizer.transform(X_test_raw).toarray()

# 4. Multinomial Naive Bayes Model from Scratch
class MultinomialNaiveBayesFromScratch:
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # Laplace smoothing

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        self.priors = np.bincount(y) / float(n_samples)
        self.feature_counts = np.zeros((n_classes, n_features))
        for c in self.classes:
            self.feature_counts[c] = X[y == c].sum(axis=0)

        self.class_word_counts = self.feature_counts.sum(axis=1)

    def predict(self, X):
        log_probs = []
        for c in self.classes:
            smoothed_word_probs = (self.feature_counts[c] + self.alpha) / (self.class_word_counts[c] + self.alpha * X.shape[1])
            log_likelihood = X @ np.log(smoothed_word_probs)
            log_prior = np.log(self.priors[c])
            log_probs.append(log_prior + log_likelihood)

        log_probs = np.array(log_probs).T
        return np.argmax(log_probs, axis=1)

# 5. Train Model
model = MultinomialNaiveBayesFromScratch(alpha=1.0)
model.fit(X_train, y_train)

# 6. Predictions & Evaluation
y_pred = model.predict(X_test)
spam_f1 = f1_score(y_test, y_pred, pos_label=1)

print("\n--- Model Evaluation ---")
print(f"F1-Score on Spam Class: {spam_f1:.4f}\n")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

--- Dataset Sample ---
  label                                               text  label_num
0   ham  Go until jurong point, crazy.. Available only ...          0
1   ham                      Ok lar... Joking wif u oni...          0
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...          1
3   ham  U dun say so early hor... U c already then say...          0
4   ham  Nah I don't think he goes to usf, he lives aro...          0

--- Model Evaluation ---
F1-Score on Spam Class: 0.9416

              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       966
        Spam       0.96      0.92      0.94       149

    accuracy                           0.98      1115
   macro avg       0.98      0.96      0.97      1115
weighted avg       0.98      0.98      0.98      1115



# Machine Learning Assignment – SMS Spam Classification
## Naive Bayes – Implementation from Scratch

| Field | Details |
| :--- | :--- |
| **Student** | Bar Revivo \| Last 4 digits of ID: 0076 |
| **Assignment Type** | Text Analysis (NLP) |
| **Learning Type** | Binary Classification |
| **Algorithm Implemented** | Multinomial Naive Bayes (Full implementation from scratch) |
| **Evaluation Metric** | F1-score on the `spam` class (Primary class in binary classification) |
| **Dataset** | [SMS Spam Collection Dataset on Kaggle](https://www.kaggle.com/datasets/hamnawaseem112222222/sms-spam-collection-5572-labeled-sms-messages) |

---

## Prompts & AI Assistance
Below are the main prompts used throughout the assignment, their purpose, and additional sources used. All code was reviewed and verified to ensure complete understanding and correctness.

1. **Prompt:** "Explain how to structure a machine learning pipeline for text classification using Naive Bayes from scratch."
   - **Purpose:** Understanding the workflow and steps required for NLP data preparation.
2. **Prompt:** "How to convert raw text messages into a word frequency matrix without data leakage?"
   - **Purpose:** Implementing Feature Engineering correctly on Train and Test sets.